# Validación del historial de incendios EGIF

Valida los eventos oficiales de EGIF, su asignación a la rejilla de 1 km y la cobertura temporal del target. No genera archivos de imagen.

In [ ]:
from pathlib import Path

import geopandas as gpd
import json
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})

EVENTS_PATH = Path("../data/processed/target/egif_events_2018_2023.gpkg")
TARGET_PATH = Path("../data/processed/target/egif_target_2018_2023.parquet")
METADATA_PATH = Path("../data/processed/target/egif_target_2018_2023_metadata.json")
GRID_PATH = Path("../data/processed/grid/grid_1km_topography_test.gpkg")

In [ ]:
events = gpd.read_file(EVENTS_PATH)
target = pd.read_parquet(TARGET_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

print(json.dumps(metadata, indent=2, ensure_ascii=False))
display(events.head())
display(target.head())

In [ ]:
assert events.crs.to_string() == "EPSG:3035"
assert set(target["target_ignicion"].unique()) == {1}
assert target["fecha"].min() >= pd.Timestamp("2018-01-01")

print(f"Eventos EGIF geolocalizados: {len(events):,}")
print(f"Positivos únicos celda-día: {len(target):,}")
print(f"Incendios agregados: {int(target['n_incendios'].sum()):,}")
print(f"Cobertura real: {target['fecha'].min():%Y-%m-%d} → {target['fecha'].max():%Y-%m-%d}")
if metadata["warning"]:
    print("AVISO:", metadata["warning"])

In [ ]:
counts = events.groupby(events["fecha"].dt.year).size()
fig, ax = plt.subplots(figsize=(8, 4.5), facecolor="white")
counts.plot.bar(ax=ax, color="#c2410c")
ax.set_title("Incendios EGIF por año")
ax.set_xlabel("Año")
ax.set_ylabel("Número de incendios")
ax.set_facecolor("white")
plt.tight_layout()
plt.show()

In [ ]:
grid = gpd.read_file(GRID_PATH)
grid = grid.loc[grid["is_galicia"] == 1]

fig, ax = plt.subplots(figsize=(8, 8), facecolor="white")
grid.boundary.plot(ax=ax, linewidth=0.08, color="lightgrey")
events.plot(ax=ax, markersize=1.5, color="#dc2626", alpha=0.55)
ax.set_title("Localización de los incendios EGIF (2018–2023)")
ax.set_axis_off()
ax.set_facecolor("white")
plt.tight_layout()
plt.show()

In [ ]:
events = None
target = None
grid = None